In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import math
import seaborn as sns
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from statsmodels.stats.proportion import proportions_ztest
from scipy import stats

In [ ]:
def fourier_performance_counts(cols_list_unmodified, gof_type_list, df_list, method_list, print_iqr_stdev, short_long, print_csv):
  cols_list = [item+f' ({gof_type})' for item in cols_list_unmodified for gof_type in gof_type_list]
  # price_cols_list = [item+f' ({gof_type})' for item in cols_list_unmodified[:-1] for gof_type in gof_type_list]
  # volume_cols_list = ['Volume'+f' ({gof_type})' for gof_type in gof_type_list]

  if method_list[0] != 'Orig' or method_list[1] != 'P-FTD':
    print('[Performance counts] INCORRECT ORDER! Put ORIG and P-FTD as first and second methods.')
    return


  median_value = []
  iqr_value = []
  log_mean_value = []
  log_stdev_value = []
  for stock in stock_list:
    for start_date in start_date_list:
      for i in range(len(df_list)):
        df = df_list[i]
        method = method_list[i]
        data = df[cols_list][(df['Stock']==stock) & (df['Start Date']==start_date)]
        median_result = data.median().tolist()
        q1 = np.array(data.quantile([0.25]))
        q3 = np.array(data.quantile([0.75]))
        iqr = q3[0] - q1[0]
        median_value.append([stock, start_date, method] + median_result)
        iqr_value.append([stock, start_date, method] + iqr.tolist())

        log_data = np.log(data)
        log_mean_result = log_data.mean().tolist()
        log_mean_value.append([stock, start_date, method] + log_mean_result)
        log_stdev_result = log_data.std().tolist()
        log_stdev_value.append([stock, start_date, method] + log_stdev_result)




  median_results = pd.DataFrame(median_value, columns = ['Stock', 'Start Date', 'Method'] + cols_list)
  iqr_results = pd.DataFrame(iqr_value, columns = ['Stock', 'Start Date', 'Method'] + cols_list)
  log_mean_results = pd.DataFrame(log_mean_value, columns = ['Stock', 'Start Date', 'Method'] + cols_list)
  log_stdev_results = pd.DataFrame(log_stdev_value, columns = ['Stock', 'Start Date', 'Method'] + cols_list)
  log_mean_results

  best_median_list = []
  best_iqr_list = []
  best_log_mean_list = []
  best_log_stdev_list = []
  for stock in stock_list:
    for start_date in start_date_list:
      best_list = []
      best_list2 = []
      best_list3 = []
      best_list4 = []
      for col in cols_list:
        df_median = median_results[(median_results['Stock']==stock) &
                                        (median_results['Start Date'] == start_date)]
        df_iqr = iqr_results[(iqr_results['Stock']==stock) &
                                        (iqr_results['Start Date'] == start_date)]
        df_log_mean = log_mean_results[(log_mean_results['Stock']==stock) &
                                        (log_mean_results['Start Date'] == start_date)]
        df_log_stdev = log_stdev_results[(log_stdev_results['Stock']==stock) &
                                        (log_stdev_results['Start Date'] == start_date)]
        # print(f'{stock} {col} (Start: {start_date}), Best performance (Median) = ', df_median.loc[df_median[col].idxmin(), 'Method'])
        # print(f'{stock} {col} (Start: {start_date}), Best performance (IQR) = ', df_iqr.loc[df_iqr[col].idxmin(), 'Method'])
        best_median = df_median.loc[df_median[col].idxmin(), 'Method']
        best_list.append(best_median)
        best_iqr = df_iqr.loc[df_iqr[col].idxmin(), 'Method']
        best_list2.append(best_iqr)
        best_log_mean = df_log_mean.loc[df_log_mean[col].idxmin(), 'Method']
        best_list3.append(best_log_mean)
        best_log_stdev = df_log_stdev.loc[df_log_stdev[col].idxmin(), 'Method']
        best_list4.append(best_log_stdev)

      best_median_list.append([stock, start_date]+ best_list)
      best_iqr_list.append([stock, start_date]+ best_list2)
      best_log_mean_list.append([stock, start_date]+ best_list3)
      best_log_stdev_list.append([stock, start_date]+ best_list4)

  best_median_list = pd.DataFrame(best_median_list, columns = ['Stock', 'Start Date']+cols_list)
  best_iqr_list = pd.DataFrame(best_iqr_list, columns = ['Stock', 'Start Date']+cols_list)
  best_log_mean_list = pd.DataFrame(best_log_mean_list, columns = ['Stock', 'Start Date']+cols_list)
  best_log_stdev_list = pd.DataFrame(best_log_stdev_list, columns = ['Stock', 'Start Date']+cols_list)


  model_type = df_list[0]['Type'].unique()

  performance_counts = pd.DataFrame(columns = ['Method'] + cols_list)

  ## ----------- Collecting the best performance counts for OHLCV: ------------------

  median_counts = best_median_list[cols_list].apply(lambda x: x.value_counts()).reset_index(names = 'Method')
  print('Median Counts:')
  print(median_counts)
  print('-'*50)
  performance_counts = pd.concat([performance_counts, pd.DataFrame([{'Method':'Median Counts'}])], ignore_index=True)
  performance_counts = pd.concat([performance_counts, median_counts], ignore_index=True)


  iqr_counts = best_iqr_list[cols_list].apply(lambda x: x.value_counts()).reset_index(names = 'Method')
  print('IQR Counts:')
  print(iqr_counts)
  print('-'*50)
  performance_counts = pd.concat([performance_counts, pd.DataFrame([{'Method':'IQR Counts'}])], ignore_index=True)
  performance_counts = pd.concat([performance_counts, iqr_counts], ignore_index=True)

  log_mean_counts = best_log_mean_list[cols_list].apply(lambda x: x.value_counts()).reset_index(names = 'Method')
  print('Log Mean Counts:')
  print(log_mean_counts)
  print('-'*50)
  performance_counts = pd.concat([performance_counts, pd.DataFrame([{'Method':'Log Mean Counts'}])], ignore_index=True)
  performance_counts = pd.concat([performance_counts, log_mean_counts], ignore_index=True)

  # if print_iqr_stdev == 'Yes':
  log_stdev_counts = best_log_stdev_list[cols_list].apply(lambda x: x.value_counts()).reset_index(names = 'Method')
  print('Log StDev Counts:')
  print(log_stdev_counts)
  print('-'*50)
  performance_counts = pd.concat([performance_counts, pd.DataFrame([{'Method':'Log StDev Counts'}])], ignore_index=True)
  performance_counts = pd.concat([performance_counts, log_stdev_counts], ignore_index=True)

  if print_csv == 'Yes':
    performance_counts.to_csv(f'{short_long}_Performance_Counts_{model_type[0]}.csv')

  return performance_counts

In [ ]:
def fourier_ratio_comparison(cols_list_unmodified, gof_type_list, df_list, method_list, short_long, print_csv):
    cols_list = [item+f' ({gof_type})' for item in cols_list_unmodified for gof_type in gof_type_list]
    price_cols_list = [item+f' ({gof_type})' for item in cols_list_unmodified[:-1] for gof_type in gof_type_list]
    volume_cols_list = [f'Volume ({gof_type})' for item in cols_list_unmodified[:-1] for gof_type in gof_type_list]

    if method_list[0] != 'Orig' or method_list[1] != 'P-FTD':
      print('[Performance ratios] INCORRECT ORDER! Put ORIG and P-FTD as first and second methods.')
      return


    model_type = df_list[0]['Type'].unique()

    median_orig_comparison = []
    median_padding_comparison = []

    iqr_orig_comparison = []
    iqr_padding_comparison = []

    log_mean_orig_comparison = []
    log_mean_padding_comparison = []

    log_stdev_orig_comparison = []
    log_stdev_padding_comparison = []


    for stock in stock_list:
      for start_date in start_date_list:
        df_orig = df_list[0]
        df_padding = df_list[1]
        orig_comp_df = df_orig[cols_list][(df_orig['Stock']==stock) & (df_orig['Start Date']==start_date)]
        padding_comp_df = df_padding[cols_list][(df_padding['Stock']==stock) & (df_padding['Start Date']==start_date)]

        ## Calculating the IQR for the original and padding data:
        q1_orig = np.array(orig_comp_df.quantile([0.25]))
        q3_orig = np.array(orig_comp_df.quantile([0.75]))
        iqr_orig = q3_orig[0] - q1_orig[0]

        q1_padding = np.array(padding_comp_df.quantile([0.25]))
        q3_padding = np.array(padding_comp_df.quantile([0.75]))
        iqr_padding = q3_padding[0] - q1_padding[0]



        i = 0
        for df in df_list[1:]:
          i = i+1
          method = method_list[i]
          performance_df = df[cols_list][(df['Stock']==stock) & (df['Start Date']==start_date)]

          ## Calculating median ratio:
          median_ratio_orig = performance_df.median()/orig_comp_df.median()
          median_ratio_padding = performance_df.median()/padding_comp_df.median()
          median_orig_comparison.append([stock, start_date, method] + median_ratio_orig.astype(float).tolist())
          median_padding_comparison.append([stock, start_date, method] + median_ratio_padding.astype(float).tolist())

          ## Calculating IQR ratio:
          q1_performance = np.array(performance_df.quantile([0.25]))
          q3_performance = np.array(performance_df.quantile([0.75]))
          iqr_performance = q3_performance[0] - q1_performance[0]
          iqr_ratio_orig = iqr_performance/iqr_orig
          iqr_ratio_padding = iqr_performance/iqr_padding
          iqr_orig_comparison.append([stock, start_date, method] + iqr_ratio_orig.astype(float).tolist())
          iqr_padding_comparison.append([stock, start_date, method] + iqr_ratio_padding.astype(float).tolist())


          ## Calculating Log-Mean ratio:
          log_mean_ratio_orig = np.array(np.log(performance_df).mean().tolist())/np.array(np.log(orig_comp_df).mean().tolist())
          log_mean_ratio_padding = np.array(np.log(performance_df).mean().tolist())/np.array(np.log(padding_comp_df).mean().tolist())
          log_mean_orig_comparison.append([stock, start_date, method] + log_mean_ratio_orig.astype(float).tolist())
          log_mean_padding_comparison.append([stock, start_date, method] + log_mean_ratio_padding.astype(float).tolist())

          ## Calculating Log-StDev ratio:
          log_stdev_ratio_orig = np.array(np.log(performance_df).std().tolist())/np.array(np.log(orig_comp_df).std().tolist())
          log_stdev_ratio_padding = np.array(np.log(performance_df).std().tolist())/np.array(np.log(padding_comp_df).std().tolist())
          log_stdev_orig_comparison.append([stock, start_date, method] + log_stdev_ratio_orig.astype(float).tolist())
          log_stdev_padding_comparison.append([stock, start_date, method] + log_stdev_ratio_padding.astype(float).tolist())

    median_orig_comparison = pd.DataFrame(median_orig_comparison, columns = ['Stock', 'Start Date', 'Method'] + cols_list)
    median_padding_comparison = pd.DataFrame(median_padding_comparison, columns = ['Stock', 'Start Date', 'Method'] + cols_list)

    iqr_orig_comparison = pd.DataFrame(iqr_orig_comparison, columns = ['Stock', 'Start Date', 'Method'] + cols_list)
    iqr_padding_comparison = pd.DataFrame(iqr_padding_comparison, columns = ['Stock', 'Start Date', 'Method'] + cols_list)

    log_mean_orig_comparison = pd.DataFrame(log_mean_orig_comparison, columns = ['Stock', 'Start Date', 'Method'] + cols_list)
    log_mean_padding_comparison = pd.DataFrame(log_mean_padding_comparison, columns = ['Stock', 'Start Date', 'Method'] + cols_list)

    log_stdev_orig_comparison = pd.DataFrame(log_stdev_orig_comparison, columns = ['Stock', 'Start Date', 'Method'] + cols_list)
    log_stdev_padding_comparison = pd.DataFrame(log_stdev_padding_comparison, columns = ['Stock', 'Start Date', 'Method'] + cols_list)


    orig_comparison_results = []
    padding_comparison_results = []

    orig_iqr_comparison_results = []
    padding_iqr_comparison_results = []

    orig_log_mean_comparison_results = []
    padding_log_mean_comparison_results = []

    orig_log_stdev_comparison_results = []
    padding_log_stdev_comparison_results = []

    for method in method_list[1:]:
      orig_ratio_data = median_orig_comparison[median_orig_comparison['Method'] == method]
      orig_comparison_results.append(['Median', method]+orig_ratio_data[cols_list].median().round(4).tolist()) #.mean()
      # padding_ratio_data = median_padding_comparison[median_padding_comparison['Method'] == method]
      # padding_comparison_results.append([method]+padding_ratio_data[cols_list].median().round(4).tolist()) #.mean()

      orig_iqr_ratio_data = iqr_orig_comparison[iqr_orig_comparison['Method'] == method]
      orig_iqr_comparison_results.append(['IQR', method]+orig_iqr_ratio_data[cols_list].median().round(4).tolist()) #.mean()
      # padding_iqr_ratio_data = iqr_padding_comparison[iqr_padding_comparison['Method'] == method]
      # padding_iqr_comparison_results.append([method]+padding_iqr_ratio_data[cols_list].median().round(4).tolist()) #.mean()

      orig_log_mean_ratio_data = log_mean_orig_comparison[log_mean_orig_comparison['Method'] == method]
      orig_log_mean_comparison_results.append(['Log Mean', method]+orig_log_mean_ratio_data[cols_list].median().round(4).tolist()) #.mean()
      # padding_log_mean_ratio_data = log_mean_padding_comparison[log_mean_padding_comparison['Method'] == method]
      # padding_log_mean_comparison_results.append([method]+padding_log_mean_ratio_data[cols_list].median().round(4).tolist()) #.mean()

      orig_log_stdev_ratio_data = log_stdev_orig_comparison[log_stdev_orig_comparison['Method'] == method]
      orig_log_stdev_comparison_results.append(['Log StDev', method]+orig_log_stdev_ratio_data[cols_list].median().round(4).tolist()) #.mean()
      # padding_log_stdev_ratio_data = log_stdev_padding_comparison[log_stdev_padding_comparison['Method'] == method]
      # padding_log_stdev_comparison_results.append([method]+padding_log_stdev_ratio_data[cols_list].median().round(4).tolist()) #.mean()

    for method in method_list[2:]:
      padding_ratio_data = median_padding_comparison[median_padding_comparison['Method'] == method]
      padding_comparison_results.append(['Median', method]+padding_ratio_data[cols_list].median().round(4).tolist()) #.mean()

      padding_iqr_ratio_data = iqr_padding_comparison[iqr_padding_comparison['Method'] == method]
      padding_iqr_comparison_results.append(['IQR', method]+padding_iqr_ratio_data[cols_list].median().round(4).tolist()) #.mean()

      padding_log_mean_ratio_data = log_mean_padding_comparison[log_mean_padding_comparison['Method'] == method]
      padding_log_mean_comparison_results.append(['Log Mean', method]+padding_log_mean_ratio_data[cols_list].median().round(4).tolist()) #.mean()

      padding_log_stdev_ratio_data = log_stdev_padding_comparison[log_stdev_padding_comparison['Method'] == method]
      padding_log_stdev_comparison_results.append(['Log StDev', method]+padding_log_stdev_ratio_data[cols_list].median().round(4).tolist()) #.mean()

    orig_median_comparison_results = pd.DataFrame(orig_comparison_results, columns = ['Type', 'Method'] + cols_list)
    padding_median_comparison_results = pd.DataFrame(padding_comparison_results, columns = ['Type', 'Method'] + cols_list)

    orig_iqr_comparison_results = pd.DataFrame(orig_iqr_comparison_results, columns = ['Type', 'Method'] + cols_list)
    padding_iqr_comparison_results = pd.DataFrame(padding_iqr_comparison_results, columns = ['Type', 'Method'] + cols_list)

    orig_log_mean_comparison_results = pd.DataFrame(orig_log_mean_comparison_results, columns = ['Type', 'Method'] + cols_list)
    padding_log_mean_comparison_results = pd.DataFrame(padding_log_mean_comparison_results, columns = ['Type', 'Method'] + cols_list)

    orig_log_stdev_comparison_results = pd.DataFrame(orig_log_stdev_comparison_results, columns = ['Type', 'Method'] + cols_list)
    padding_log_stdev_comparison_results = pd.DataFrame(padding_log_stdev_comparison_results, columns = ['Type', 'Method'] + cols_list)


    ## Collecting all orig ratio data into CSV:
    orig_ratio_total_df = pd.DataFrame(columns = ['Type', 'Method'] + cols_list)

    orig_ratio_total_df = pd.concat([orig_ratio_total_df, orig_median_comparison_results], ignore_index=True)


    orig_ratio_total_df = pd.concat([orig_ratio_total_df, orig_iqr_comparison_results], ignore_index=True)


    orig_ratio_total_df = pd.concat([orig_ratio_total_df, orig_log_mean_comparison_results], ignore_index=True)


    orig_ratio_total_df = pd.concat([orig_ratio_total_df, orig_log_stdev_comparison_results], ignore_index=True)



    ## Collecting all padding ratio data into CSV:
    padding_ratio_total_df = pd.DataFrame( columns = ['Type', 'Method'] + cols_list)

    padding_ratio_total_df = pd.concat([padding_ratio_total_df, padding_median_comparison_results], ignore_index=True)


    padding_ratio_total_df = pd.concat([padding_ratio_total_df, padding_iqr_comparison_results], ignore_index=True)


    padding_ratio_total_df = pd.concat([padding_ratio_total_df, padding_log_mean_comparison_results], ignore_index=True)


    padding_ratio_total_df = pd.concat([padding_ratio_total_df, padding_log_stdev_comparison_results], ignore_index=True)

    if print_csv == 'Yes':
      orig_ratio_total_df.to_csv(f'{short_long}_Orig_Comparison_Ratio_{model_type[0]}.csv')
      padding_ratio_total_df.to_csv(f'{short_long}_Padding_Comparison_Ratio_{model_type[0]}.csv')


    print('')
    print('')
    print('^'*40)
    print('Orig Ratio Comparison (Median):')
    print(orig_median_comparison_results)
    print('')
    print('')
    print('Padding Ratio Comparison:')
    print(padding_median_comparison_results)
    print('')
    print('')
    print('^'*40)

    print('Orig Ratio Comparison (IQR):')
    print(orig_iqr_comparison_results)
    print('')
    print('')
    print('Padding Ratio Comparison (IQR):')
    print(padding_iqr_comparison_results)
    print('')
    print('')
    print('^'*40)

    print('Orig Ratio Comparison (Log-Mean):')
    print(orig_log_mean_comparison_results)
    print('')
    print('')
    print('Padding Ratio Comparison (Log-Mean):')
    print(padding_log_mean_comparison_results)
    print('')
    print('')
    print('^'*40)

    print('Orig Ratio Comparison (Log-StDev):')
    print(orig_log_stdev_comparison_results)
    print('')
    print('')
    print('Padding Ratio Comparison (Log-StDev):')
    print(padding_log_stdev_comparison_results)
    print('')
    print('')
    print('^'*40)

    return padding_ratio_total_df

In [ ]:
def statistical_performance_analysis(cols_list_unmodified, gof_type_list, df_list, method_list, short_long, print_csv):
  cols_list = [item+f' ({gof_type})' for item in cols_list_unmodified for gof_type in gof_type_list]

  model_type = df_list[0]['Type'].unique()

  if method_list[0] != 'Orig' or method_list[1] != 'P-FTD':
    print('[Wilcoxon analysis] INCORRECT ORDER! Put ORIG and P-FTD as first and second methods.')
    return

  median_value = []
  iqr_value = []
  log_mean_value = []
  log_stdev_value = []
  for stock in stock_list:
    for start_date in start_date_list:
      for i in range(len(df_list)):
        df = df_list[i]
        df['Stock-Date'] = df[['Stock', 'Start Date', 'End Date', 'Seed']].apply( lambda x:' '.join(x.astype(str)), axis=1)
        method = method_list[i]
        data = df[cols_list][(df['Stock']==stock) & (df['Start Date']==start_date)]
        median_result = data.median().tolist()
        q1 = np.array(data.quantile([0.25]))
        q3 = np.array(data.quantile([0.75]))
        iqr = q3[0] - q1[0]
        median_value.append([stock, start_date, method] + median_result)
        iqr_value.append([stock, start_date, method] + iqr.tolist())

        log_data = np.log(data)
        log_mean_result = log_data.mean().tolist()
        log_mean_value.append([stock, start_date, method] + log_mean_result)
        log_stdev_result = log_data.std().tolist()
        log_stdev_value.append([stock, start_date, method] + log_stdev_result)

  median_results = pd.DataFrame(median_value, columns = ['Stock', 'Start Date', 'Method'] + cols_list)
  iqr_results = pd.DataFrame(iqr_value, columns = ['Stock', 'Start Date', 'Method'] + cols_list)
  log_mean_results = pd.DataFrame(log_mean_value, columns = ['Stock', 'Start Date', 'Method'] + cols_list)
  log_stdev_results = pd.DataFrame(log_stdev_value, columns = ['Stock', 'Start Date', 'Method'] + cols_list)

  median_results['Stock-Date'] = median_results[['Stock', 'Start Date']].apply( lambda x:' '.join(x.astype(str)), axis=1)
  iqr_results['Stock-Date'] = iqr_results[['Stock', 'Start Date']].apply( lambda x:' '.join(x.astype(str)), axis=1)
  log_mean_results['Stock-Date'] = log_mean_results[['Stock', 'Start Date']].apply( lambda x:' '.join(x.astype(str)), axis=1)
  log_stdev_results['Stock-Date'] = log_stdev_results[['Stock', 'Start Date']].apply( lambda x:' '.join(x.astype(str)), axis=1)



  results_df_list = [median_results, iqr_results, log_mean_results, log_stdev_results]
  results_name_list = ['Median', 'IQR', 'Log Mean', 'Log StDev']

  wilx_stat_orig_total_results = []
  wilx_p_value_orig_total_results = []
  wilx_stat_padding_total_results = []
  wilx_p_value_padding_total_results = []

  results_name_idx = -1
  for results_df in results_df_list:
    results_name_idx = results_name_idx + 1
    results_orig = results_df[results_df['Method']==method_list[0]].reset_index(drop=True)
    results_padding = results_df[results_df['Method']==method_list[1]].reset_index(drop=True)
    for method in method_list[1:]:
      results_method = results_df[results_df['Method']==method].reset_index(drop=True)
      orig_match = results_method['Stock-Date'].equals(results_orig['Stock-Date'])
      padding_match = results_method['Stock-Date'].equals(results_padding['Stock-Date'])
      print('')
      print('-'*50)
      print(f'METHOD: {method} ({results_name_list[results_name_idx]})')
      print(f'Orig Match: {orig_match}, Padding Match: {padding_match}')
      print('')

      wilx_stat_orig_results = [results_name_list[results_name_idx], method]
      wilx_p_value_orig_results = [results_name_list[results_name_idx], method]
      wilx_stat_padding_results = [results_name_list[results_name_idx], method]
      wilx_p_value_padding_results = [results_name_list[results_name_idx], method]

      for col in cols_list:
        wilx_stat_orig, p_value_wilx_orig = stats.wilcoxon(results_method[col], results_orig[col], alternative='two-sided')
        if p_value_wilx_orig <= 0.01:
          sig_label_orig = ' **'
        elif p_value_wilx_orig <= 0.05:
          sig_label_orig = ' *'
        else:
          sig_label_orig = ''

        print(f'Wilcoxon Signed Rank Test Results Column: {col} ({results_name_list[results_name_idx]}) Orig Statistic = {wilx_stat_orig:.4f}, p-value = {p_value_wilx_orig:.4f}) {sig_label_orig}')
        wilx_stat_orig_results.append(round(wilx_stat_orig,4).astype(str)+sig_label_orig)
        wilx_p_value_orig_results.append(p_value_wilx_orig) #.astype(str)+sig_label_orig)

        if method != method_list[1]:
          wilx_stat_padding, p_value_wilx_padding = stats.wilcoxon(results_method[col], results_padding[col], alternative='two-sided')
          if p_value_wilx_padding <= 0.01:
            sig_label_padding = ' **'
          elif p_value_wilx_padding <= 0.05:
            sig_label_padding = ' *'
          else:
            sig_label_padding = ''


          wilx_stat_padding_results.append(round(wilx_stat_padding,4).astype(str)+sig_label_padding)
          wilx_p_value_padding_results.append(p_value_wilx_padding)


          print(f'Wilcoxon Signed Rank Test Results Column: {col} ({results_name_list[results_name_idx]}) Padding Statistic = {wilx_stat_padding:.4f}, p-value = {p_value_wilx_padding:.4f}) {sig_label_padding}')

      wilx_stat_orig_total_results.append(wilx_stat_orig_results)
      wilx_p_value_orig_total_results.append(wilx_p_value_orig_results)
      wilx_stat_padding_total_results.append(wilx_stat_padding_results)
      wilx_p_value_padding_total_results.append(wilx_p_value_padding_results)

  wilx_stat_orig_total_results = pd.DataFrame(wilx_stat_orig_total_results, columns = ['Type', 'Method']+cols_list)
  wilx_p_value_orig_total_results = pd.DataFrame(wilx_p_value_orig_total_results, columns = ['Type', 'Method']+cols_list)
  wilx_stat_padding_total_results = pd.DataFrame(wilx_stat_padding_total_results, columns = ['Type', 'Method']+cols_list)
  wilx_p_value_padding_total_results = pd.DataFrame(wilx_p_value_padding_total_results, columns = ['Type', 'Method']+cols_list)


  if print_csv == 'Yes':
    wilx_stat_orig_total_results.to_csv(f'{short_long}_Orig_Wilcoxon_Stat_Results_{model_type[0]}.csv')
    wilx_p_value_orig_total_results.to_csv(f'{short_long}_Orig_Wilcoxon_p_value_Results_{model_type[0]}.csv')
    wilx_stat_padding_total_results.to_csv(f'{short_long}_Padding_Wilcoxon_Stat_Results_{model_type[0]}.csv')
    wilx_p_value_padding_total_results.to_csv(f'{short_long}_Padding_Wilcoxon_p_value_Results_{model_type[0]}.csv')




In [ ]:
cols_list_unmodified = ['Open', 'High', 'Low', 'Close', 'Volume']
gof_type_list = ['RMSE', 'MAE', 'MAPE'] #, 'SDAPE']

## ANN:

In [ ]:
path_to_data = 'https://raw.githubusercontent.com/FFTStockPrediction/Fourier_Transform_Denoising_Stock_Prediction/refs/heads/main/ANN_Results/'

short_orig_ann = pd.read_csv(path_to_data+'Short_ANN_Orig_Results.csv').iloc[:, 1:]
short_p_ftd_ann = pd.read_csv(path_to_data+'Short_ANN_P_FTD_Results.csv').iloc[:, 1:]
short_exp_vd_ann = pd.read_csv(path_to_data+'Short_ANN_Exp_VD_Results.csv').iloc[:, 1:]
short_ldd_ann = pd.read_csv(path_to_data+'Short_ANN_LDD_Results.csv').iloc[:, 1:]
short_exp_ldd_ann = pd.read_csv(path_to_data+'Short_ANN_Exp_LDD_Results.csv').iloc[:, 1:]

long_orig_ann = pd.read_csv(path_to_data+'Long_ANN_Orig_Results.csv').iloc[:, 1:]
long_p_ftd_ann = pd.read_csv(path_to_data+'Long_ANN_P_FTD_Results.csv').iloc[:, 1:]
long_exp_vd_ann = pd.read_csv(path_to_data+'Long_ANN_Exp_VD_Results.csv').iloc[:, 1:]
long_ldd_ann = pd.read_csv(path_to_data+'Long_ANN_LDD_Results.csv').iloc[:, 1:]
long_exp_ldd_ann = pd.read_csv(path_to_data+'Long_ANN_Exp_LDD_Results.csv').iloc[:, 1:]

short_orig_ann

In [ ]:
stock_list = short_orig_ann['Stock'].unique().tolist()
start_date_list = short_orig_ann['Start Date'].unique().tolist()
stock_list

In [ ]:
cols_list = ['Open', 'High', 'Low', 'Close', 'Volume']

df_list = [short_orig_ann, short_p_ftd_ann,
          short_ldd_ann, short_exp_ldd_ann, short_exp_vd_ann]
method_list = ['Orig', 'P-FTD', 'LDD', 'Exp-LDD', 'Exp-VD']

short_long = 'Short'
print_csv = 'Yes'

fourier_performance_counts(cols_list_unmodified, gof_type_list, df_list, method_list, 'Yes', short_long, print_csv)
fourier_ratio_comparison(cols_list_unmodified, gof_type_list, df_list, method_list, short_long, print_csv)
statistical_performance_analysis(cols_list_unmodified, gof_type_list, df_list, method_list, short_long, print_csv)

In [ ]:
cols_list = ['Open', 'High', 'Low', 'Close', 'Volume']

df_list = [long_orig_ann, long_p_ftd_ann,
          long_ldd_ann, long_exp_ldd_ann, long_exp_vd_ann]
method_list = ['Orig', 'P-FTD', 'LDD', 'Exp-LDD', 'Exp-VD']

short_long = 'Long'
print_csv = 'Yes'

fourier_performance_counts(cols_list_unmodified, gof_type_list, df_list, method_list, 'Yes', short_long, print_csv)
fourier_ratio_comparison(cols_list_unmodified, gof_type_list, df_list, method_list, short_long, print_csv)
statistical_performance_analysis(cols_list_unmodified, gof_type_list, df_list, method_list, short_long, print_csv)

##LSTM:

In [ ]:
path_to_data = 'https://raw.githubusercontent.com/FFTStockPrediction/Fourier_Transform_Denoising_Stock_Prediction/refs/heads/main/LSTM_Results/'

short_orig_lstm = pd.read_csv(path_to_data+'Short_LSTM_Orig_Results.csv').iloc[:, 1:]
short_p_ftd_lstm = pd.read_csv(path_to_data+'Short_LSTM_P_FTD_Results.csv').iloc[:, 1:]
short_exp_vd_lstm = pd.read_csv(path_to_data+'Short_LSTM_Exp_VD_Results.csv').iloc[:, 1:]
short_ldd_lstm = pd.read_csv(path_to_data+'Short_LSTM_LDD_Results.csv').iloc[:, 1:]
short_exp_ldd_lstm = pd.read_csv(path_to_data+'Short_LSTM_Exp_LDD_Results.csv').iloc[:, 1:]

long_orig_lstm = pd.read_csv(path_to_data+'Long_LSTM_Orig_Results.csv').iloc[:, 1:]
long_p_ftd_lstm = pd.read_csv(path_to_data+'Long_LSTM_P_FTD_Results.csv').iloc[:, 1:]
long_exp_vd_lstm = pd.read_csv(path_to_data+'Long_LSTM_Exp_VD_Results.csv').iloc[:, 1:]
long_ldd_lstm = pd.read_csv(path_to_data+'Long_LSTM_LDD_Results.csv').iloc[:, 1:]
long_exp_ldd_lstm = pd.read_csv(path_to_data+'Long_LSTM_Exp_LDD_Results.csv').iloc[:, 1:]


short_orig_lstm

In [ ]:
stock_list = short_orig_lstm['Stock'].unique().tolist()
start_date_list = short_orig_lstm['Start Date'].unique().tolist()
stock_list

In [ ]:
cols_list = ['Open', 'High', 'Low', 'Close', 'Volume']

df_list = [short_orig_lstm, short_p_ftd_lstm,
          short_ldd_lstm, short_exp_ldd_lstm, short_exp_vd_lstm]
method_list = ['Orig', 'P-FTD', 'LDD', 'Exp-LDD', 'Exp-VD']

short_long = 'Short'
print_csv = 'Yes'

fourier_performance_counts(cols_list_unmodified, gof_type_list, df_list, method_list, 'Yes', short_long, print_csv)
fourier_ratio_comparison(cols_list_unmodified, gof_type_list, df_list, method_list, short_long, print_csv)
statistical_performance_analysis(cols_list_unmodified, gof_type_list, df_list, method_list, short_long, print_csv)

In [ ]:
cols_list = ['Open', 'High', 'Low', 'Close', 'Volume']

df_list = [long_orig_lstm, long_p_ftd_lstm,
          long_ldd_lstm, long_exp_ldd_lstm, long_exp_vd_lstm]
method_list = ['Orig', 'P-FTD', 'LDD', 'Exp-LDD', 'Exp-VD']

short_long = 'Long'
print_csv = 'Yes'

fourier_performance_counts(cols_list_unmodified, gof_type_list, df_list, method_list, 'Yes', short_long, print_csv)
fourier_ratio_comparison(cols_list_unmodified, gof_type_list, df_list, method_list, short_long, print_csv)
statistical_performance_analysis(cols_list_unmodified, gof_type_list, df_list, method_list, short_long, print_csv)

## GRU:

In [ ]:
path_to_data = 'https://raw.githubusercontent.com/FFTStockPrediction/Fourier_Transform_Denoising_Stock_Prediction/refs/heads/main/GRU_Results/'

short_orig_gru = pd.read_csv(path_to_data+'Short_GRU_Orig_Results.csv').iloc[:, 1:]
short_p_ftd_gru = pd.read_csv(path_to_data+'Short_GRU_P_FTD_Results.csv').iloc[:, 1:]
short_exp_vd_gru = pd.read_csv(path_to_data+'Short_GRU_Exp_VD_Results.csv').iloc[:, 1:]
short_ldd_gru = pd.read_csv(path_to_data+'Short_GRU_LDD_Results.csv').iloc[:, 1:]
short_exp_ldd_gru = pd.read_csv(path_to_data+'Short_GRU_Exp_LDD_Results.csv').iloc[:, 1:]

long_orig_gru = pd.read_csv(path_to_data+'Long_GRU_Orig_Results.csv').iloc[:, 1:]
long_p_ftd_gru = pd.read_csv(path_to_data+'Long_GRU_P_FTD_Results.csv').iloc[:, 1:]
long_exp_vd_gru = pd.read_csv(path_to_data+'Long_GRU_Exp_VD_Results.csv').iloc[:, 1:]
long_ldd_gru = pd.read_csv(path_to_data+'Long_GRU_LDD_Results.csv').iloc[:, 1:]
long_exp_ldd_gru = pd.read_csv(path_to_data+'Long_GRU_Exp_LDD_Results.csv').iloc[:, 1:]


short_orig_gru

In [ ]:
stock_list = short_orig_gru['Stock'].unique().tolist()
start_date_list = short_orig_gru['Start Date'].unique().tolist()
stock_list

In [ ]:
cols_list = ['Open', 'High', 'Low', 'Close', 'Volume']

df_list = [short_orig_gru, short_p_ftd_gru,
          short_ldd_gru, short_exp_ldd_gru, short_exp_vd_gru]
method_list = ['Orig', 'P-FTD', 'LDD', 'Exp-LDD', 'Exp-VD']

short_long = 'Short'
print_csv = 'Yes'

fourier_performance_counts(cols_list_unmodified, gof_type_list, df_list, method_list, 'Yes', short_long, print_csv)
fourier_ratio_comparison(cols_list_unmodified, gof_type_list, df_list, method_list, short_long, print_csv)
statistical_performance_analysis(cols_list_unmodified, gof_type_list, df_list, method_list, short_long, print_csv)

In [ ]:
cols_list = ['Open', 'High', 'Low', 'Close', 'Volume']

df_list = [long_orig_gru, long_p_ftd_gru,
          long_ldd_gru, long_exp_ldd_gru, long_exp_vd_gru]
method_list = ['Orig', 'P-FTD', 'LDD', 'Exp-LDD', 'Exp-VD']

short_long = 'Long'
print_csv = 'Yes'

fourier_performance_counts(cols_list_unmodified, gof_type_list, df_list, method_list, 'Yes', short_long, print_csv)
fourier_ratio_comparison(cols_list_unmodified, gof_type_list, df_list, method_list, short_long, print_csv)
statistical_performance_analysis(cols_list_unmodified, gof_type_list, df_list, method_list, short_long, print_csv)

## Transformer:

In [ ]:
path_to_data = 'https://raw.githubusercontent.com/FFTStockPrediction/Fourier_Transform_Denoising_Stock_Prediction/refs/heads/main/Transformer_Results/'

short_orig_trans = pd.read_csv(path_to_data+'Short_Transformer_Orig_Results.csv').iloc[:, 1:]
short_p_ftd_trans = pd.read_csv(path_to_data+'Short_Transformer_P_FTD_Results.csv').iloc[:, 1:]
short_exp_vd_trans = pd.read_csv(path_to_data+'Short_Transformer_Exp_VD_Results.csv').iloc[:, 1:]
short_ldd_trans = pd.read_csv(path_to_data+'Short_Transformer_LDD_Results.csv').iloc[:, 1:]
short_exp_ldd_trans = pd.read_csv(path_to_data+'Short_Transformer_Exp_LDD_Results.csv').iloc[:, 1:]

long_orig_trans = pd.read_csv(path_to_data+'Long_Transformer_Orig_Results.csv').iloc[:, 1:]
long_p_ftd_trans = pd.read_csv(path_to_data+'Long_Transformer_P_FTD_Results.csv').iloc[:, 1:]
long_exp_vd_trans = pd.read_csv(path_to_data+'Long_Transformer_Exp_VD_Results.csv').iloc[:, 1:]
long_ldd_trans = pd.read_csv(path_to_data+'Long_Transformer_LDD_Results.csv').iloc[:, 1:]
long_exp_ldd_trans = pd.read_csv(path_to_data+'Long_Transformer_Exp_LDD_Results.csv').iloc[:, 1:]


short_ldd_trans

In [ ]:
stock_list = short_exp_ldd_trans['Stock'].unique().tolist()
start_date_list = short_exp_ldd_trans['Start Date'].unique().tolist()
stock_list

In [ ]:
cols_list = ['Open', 'High', 'Low', 'Close', 'Volume']


df_list = [short_orig_trans, short_p_ftd_trans,
          short_ldd_trans, short_exp_ldd_trans, short_exp_vd_trans]
method_list = ['Orig', 'P-FTD', 'LDD', 'Exp-LDD', 'Exp-VD']

short_long = 'Short'
print_csv = 'Yes'

fourier_performance_counts(cols_list_unmodified, gof_type_list, df_list, method_list, 'Yes', short_long, print_csv)
fourier_ratio_comparison(cols_list_unmodified, gof_type_list, df_list, method_list, short_long, print_csv)
statistical_performance_analysis(cols_list_unmodified, gof_type_list, df_list, method_list, short_long, print_csv)

In [ ]:
cols_list = ['Open', 'High', 'Low', 'Close', 'Volume']


df_list = [long_orig_trans, long_p_ftd_trans,
          long_ldd_trans, long_exp_ldd_trans, long_exp_vd_trans]
method_list = ['Orig', 'P-FTD', 'LDD', 'Exp-LDD', 'Exp-VD']

short_long = 'Long'
print_csv = 'Yes'

fourier_performance_counts(cols_list_unmodified, gof_type_list, df_list, method_list, 'Yes', short_long, print_csv)
fourier_ratio_comparison(cols_list_unmodified, gof_type_list, df_list, method_list, short_long, print_csv)
statistical_performance_analysis(cols_list_unmodified, gof_type_list, df_list, method_list, short_long, print_csv)